In [43]:
from transformers import AutoModelForSequenceClassification

checkpoint = "prajjwal1/bert-tiny"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

print(model)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 128, padding_idx=0)
      (position_embeddings): Embedding(512, 128)
      (token_type_embeddings): Embedding(2, 128)
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-1): 2 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=128, out_features=128, bias=True)
              (key): Linear(in_features=128, out_features=128, bias=True)
              (value): Linear(in_features=128, out_features=128, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=128, out_features=128, bias=True)
              (LayerNorm): LayerNorm((128,), eps=1e-1

In [59]:
checkpoint = "DeepWokLab/bert-tiny"
tokenizer_checkpoint = "DeepWokLab/bert-tiny"
dataset_name = "imdb"

from chop.tools import get_tokenized_dataset

dataset, tokenizer = get_tokenized_dataset(
    dataset=dataset_name,
    checkpoint=tokenizer_checkpoint,
    return_tokenizer=True,
)

INFO     Tokenizing dataset imdb with AutoTokenizer for DeepWokLab/bert-tiny.


In [60]:
from chop import MaseGraph
import chop.passes as passes

model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
model.config.problem_type = "single_label_classification"

mg = MaseGraph(
    model,
    hf_input_names=[
        "input_ids",
        "attention_mask",
        "labels",
    ],
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at DeepWokLab/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
`past_key_values` were not specified as input names, but model.config.use_cache = True. Setting model.config.use_cache = False.


In [61]:
for param in mg.model.bert.embeddings.parameters():
    param.requires_grad = False

from chop.tools import get_trainer

trainer = get_trainer(
    model=mg.model,
    tokenized_dataset=dataset,
    tokenizer=tokenizer,
    evaluate_metric="accuracy",
)

trainer.train()
eval_results = trainer.evaluate()
print(f"Evaluation accuracy: {eval_results['eval_accuracy']}")

/vol/bitbucket/hv122/adls-project/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


OutOfMemoryError: CUDA out of memory. Tried to allocate 16.00 MiB. GPU 0 has a total capacity of 15.57 GiB of which 15.00 MiB is free. Process 153301 has 5.21 GiB memory in use. Process 1346501 has 626.00 MiB memory in use. Including non-PyTorch memory, this process has 5.26 GiB memory in use. Process 1404045 has 4.36 GiB memory in use. Of the allocated memory 4.83 GiB is allocated by PyTorch, and 144.51 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [62]:
import torch

def replace_gelu_with_relu_transform_pass(mg, pass_args=None):
    nodes_to_replace = []

    for node in mg.fx_graph.nodes:
        if node.op == "call_function" and getattr(node.target, "__name__", "") == "gelu":
            nodes_to_replace.append(node)

    for node in nodes_to_replace:
        with mg.fx_graph.inserting_after(node):
            # relu_node = mg.fx_graph.call_function(
            #     torch.relu,
            #     args=(node.args[0],),
            # )
            # relu_node.name = node.name.replace("gelu", "relu")
            # node.replace_all_uses_with(relu_node)
            node.replace_all_uses_with(node.args[0])  # bypass: wire input directly to consumers

        mg.fx_graph.erase_node(node)

    mg.fx_graph.lint()
    print(f"Successfully removed {len(nodes_to_replace)} GeLU layers (passthrough mode).")
    return mg, {}

replace_gelu_with_relu_transform_pass(mg)

Successfully removed 2 GeLU layers (passthrough mode).


(<chop.ir.graph.mase_graph.MaseGraph at 0x7edb54a758d0>, {})

In [63]:
gelu_remaining = sum(
    1 for node in mg.fx_graph.nodes
    if node.op == "call_function" and getattr(node.target, "__name__", "") == "gelu"
)
relu_added = sum(
    1 for node in mg.fx_graph.nodes
    if node.op == "call_function" and getattr(node.target, "__name__", "") == "relu"
)
print(f"GELU nodes remaining: {gelu_remaining}")
print(f"ReLU nodes present:   {relu_added}")

GELU nodes remaining: 0
ReLU nodes present:   0


In [64]:
from chop.tools import get_trainer

print("\nEvaluating Destroyed Model (ReLU)...")
destroyed_trainer = get_trainer(
    model=mg.model, 
    tokenized_dataset=dataset,
    tokenizer=tokenizer,
    evaluate_metric="accuracy",
)
destroyed_results = destroyed_trainer.evaluate()
print(f"Destroyed Accuracy: {destroyed_results['eval_accuracy']}")
# print(f"Accuracy Drop: {baseline_results['eval_accuracy'] - destroyed_results['eval_accuracy']:.4f}")


print("\nInitiating Recovery Fine-Tuning...")
destroyed_trainer.train()
recovered_results = destroyed_trainer.evaluate()
print(f"Recovered Accuracy: {recovered_results['eval_accuracy']}")


Evaluating Destroyed Model (ReLU)...


/vol/bitbucket/hv122/adls-project/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


OutOfMemoryError: CUDA out of memory. Tried to allocate 16.00 MiB. GPU 0 has a total capacity of 15.57 GiB of which 15.00 MiB is free. Process 153301 has 5.21 GiB memory in use. Process 1346501 has 626.00 MiB memory in use. Including non-PyTorch memory, this process has 5.26 GiB memory in use. Process 1404045 has 4.36 GiB memory in use. Of the allocated memory 4.83 GiB is allocated by PyTorch, and 144.51 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

# Bert Base

In [52]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased")

print(model)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [53]:
checkpoint = "textattack/bert-base-uncased-imdb"
tokenizer_checkpoint = "textattack/bert-base-uncased-imdb"
dataset_name = "imdb"

from chop.tools import get_tokenized_dataset

dataset, tokenizer = get_tokenized_dataset(
    dataset=dataset_name,
    checkpoint=tokenizer_checkpoint,
    return_tokenizer=True,
)

INFO     Tokenizing dataset imdb with AutoTokenizer for textattack/bert-base-uncased-imdb.


In [54]:
from chop import MaseGraph
import chop.passes as passes

model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
model.config.problem_type = "single_label_classification"

mg = MaseGraph(
    model,
    hf_input_names=[
        "input_ids",
        "attention_mask",
        "labels",
    ],
)

for param in mg.model.bert.embeddings.parameters():
    param.requires_grad = False

from chop.tools import get_trainer

trainer = get_trainer(
    model=mg.model,
    tokenized_dataset=dataset,
    tokenizer=tokenizer,
    evaluate_metric="accuracy",
)

trainer.train()
eval_results = trainer.evaluate()
print(f"Evaluation accuracy: {eval_results['eval_accuracy']}")

`past_key_values` were not specified as input names, but model.config.use_cache = True. Setting model.config.use_cache = False.
/vol/bitbucket/hv122/adls-project/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,0.235900
1000,0.234800
1500,0.200400
2000,0.164900
2500,0.141300


KeyboardInterrupt: 

In [ ]:
from chop.tools import get_trainer

baseline_trainer = get_trainer(
    model=mg.model,
    tokenized_dataset=dataset,
    tokenizer=tokenizer,
    evaluate_metric="accuracy",
)

print("\nExecuting GELU -> ReLU Swap...")
mg = replace_gelu_with_relu_transform_pass(mg)

mg.model = fx.GraphModule(mg.model, mg.fx_graph)


Executing GELU -> ReLU Swap...
{'replaced_count': 0}


/vol/bitbucket/hv122/adls-project/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
print("\nEvaluating Destroyed Model (ReLU)...")
destroyed_trainer = get_trainer(
    model=mg.model, 
    tokenized_dataset=dataset,
    tokenizer=tokenizer,
    evaluate_metric="accuracy",
)
destroyed_results = destroyed_trainer.evaluate()
print(f"Destroyed Accuracy: {destroyed_results['eval_accuracy']:.4f}")
print(f"Accuracy Drop: {baseline_results['eval_accuracy'] - destroyed_results['eval_accuracy']:.4f}")


print("\nInitiating Recovery Fine-Tuning...")
destroyed_trainer.train()
recovered_results = destroyed_trainer.evaluate()
print(f"Recovered Accuracy: {recovered_results['eval_accuracy']:.4f}")

Evaluating Baseline (GELU)...


/vol/bitbucket/hv122/adls-project/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Baseline Accuracy: 0.9026

Executing GELU -> ReLU Swap...

Evaluating Destroyed Model (ReLU)...


/vol/bitbucket/hv122/adls-project/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Destroyed Accuracy: 0.9026
Accuracy Drop: 0.0000

Initiating Recovery Fine-Tuning...


Step,Training Loss


KeyboardInterrupt: 